In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from PIL import Image
import torchvision.transforms as transforms
import torchvision.models as models
import copy
# 1. Cấu hình thiết bị (GPU nếu có)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
imsize = 400 if torch.cuda.is_available() else 128  # Kích thước ảnh
# 2. Tiền xử lý ảnh (Preprocessing)
loader = transforms.Compose([
    transforms.Resize((imsize, imsize)),
    transforms.ToTensor(),
    # Chuẩn hóa theo thông số của VGG19
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [ ]:
def image_loader(image_name):
    image = Image.open(image_name).convert('RGB')
    image = loader(image).unsqueeze(0)
    return image.to(device, torch.float)
# Post-processing: Chuyển tensor back về ảnh để hiển thị
unloader = transforms.Compose([
    transforms.Normalize(mean=[-0.485/0.229, -0.456/0.224, -0.406/0.225], 
                         std=[1/0.229, 1/0.224, 1/0.225]),
    transforms.ToPILImage()
])

In [ ]:
class ContentLoss(nn.Module):
    def __init__(self, target):
        super(ContentLoss, self).__init__()
        self.target = target.detach()
    def forward(self, x):
        self.loss = nn.functional.mse_loss(x, self.target)
        return x
def gram_matrix(input):
    a, b, c, d = input.size() 
    features = input.view(a * b, c * d)
    G = torch.mm(features, features.t()) 
    return G.div(a * b * c * d)

class StyleLoss(nn.Module):
    def __init__(self, target_feature):
        super(StyleLoss, self).__init__()
        self.target = gram_matrix(target_feature).detach()
    def forward(self, x):
        G = gram_matrix(x)
        self.loss = nn.functional.mse_loss(G, self.target)
        return x

In [ ]:
vgg = models.vgg19(weights=models.VGG19_Weights.IMAGENET1K_V1).features.to(device).eval()
# 5. Xây dựng mô hình NST
# Theo ví dụ Keras: dùng block5_conv2 cho Content, và các block_conv1 cho Style
content_layers = ['conv_10'] # block4_conv2 tương đương tầng 10-12
style_layers = ['conv_1', 'conv_3', 'conv_5', 'conv_9', 'conv_13'] 
def get_style_model_and_losses(vgg, style_img, content_img):
    content_losses = []
    style_losses = []
    
    model = nn.Sequential().to(device)
    i = 0
    for layer in vgg.children():
        if isinstance(layer, nn.Conv2d):
            i += 1
            name = f'conv_{i}'
        elif isinstance(layer, nn.ReLU):
            name = f'relu_{i}'
            layer = nn.ReLU(inplace=False)
        elif isinstance(layer, nn.MaxPool2d):
            name = f'pool_{i}'
        else:
            name = 'other'
        model.add_module(name, layer)
        if name in content_layers:
            target = model(content_img).detach()
            content_loss = ContentLoss(target)
            model.add_module(f"content_loss_{i}", content_loss)
            content_losses.append(content_loss)
        if name in style_layers:
            target_feature = model(style_img).detach()
            style_loss = StyleLoss(target_feature)
            model.add_module(f"style_loss_{i}", style_loss)
            style_losses.append(style_loss)
    # Cắt bỏ các tầng thừa sau tầng loss cuối cùng
    for i in range(len(model) - 1, -1, -1):
        if isinstance(model[i], ContentLoss) or isinstance(model[i], StyleLoss):
            break
    model = model[:(i + 1)]
    return model, style_losses, content_losses
# 6. Vòng lặp tối ưu (Optimization)

In [ ]:
# 6. Vòng lặp tối ưu (Optimization)
def run_style_transfer(vgg, content_img, style_img, input_img, num_steps=300,
                       style_weight=1000000, content_weight=1):
    
    model, style_losses, content_losses = get_style_model_and_losses(
        vgg, style_img, content_img)
    
    # Chúng ta tối ưu trực tiếp trên các pixel của ảnh input
    optimizer = optim.LBFGS([input_img.requires_grad_()])
    run = [0]
    while run[0] <= num_steps:
        def closure():
            # Ràng buộc giá trị pixel trong khoảng [0, 1] sau khi Normalize là khó, 
            # nên ta thường kẹp giá trị ảnh ở cuối vòng lặp.
            input_img.data.clamp_(0, 1) # Tùy chọn nếu không Normalize
            optimizer.zero_grad()
            model(input_img)
            
            s_loss = 0
            c_loss = 0
            for sl in style_losses: s_loss += sl.loss
            for cl in content_losses: c_loss += cl.loss
            loss = s_loss * style_weight + c_loss * content_weight
            loss.backward()
            run[0] += 1
            if run[0] % 50 == 0:
                print(f"Step {run[0]}: Style Loss: {s_loss.item():4f} Content Loss: {c_loss.item():4f}")
            return loss
        optimizer.step(closure)
    input_img.data.clamp_(0, 1)
    return input_img
# 7. Thực thi
# content_img = image_loader("path_to_content.jpg")
# style_img = image_loader("path_to_style.jpg")
# input_img = content_img.clone() # Bắt đầu từ ảnh nội dung
# output = run_style_transfer(vgg, content_img, style_img, input_img)
# result_image = unloader(output.squeeze(0).cpu())
# result_image.save("output.jpg")